In [0]:
from pyspark.sql.functions import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
df = spark.table("gold_byma.fact_operaciones")
df_fact = spark.table("gold_byma.fact_operaciones")
df_dolar = spark.table("silver_byma.fecha_dolar")
dim_inst = spark.table("gold_byma.dim_instrumento").select("instrumento_sk", "simbolo")
dim_fecha = spark.table("gold_byma.dim_fecha").select("fecha_sk", "fecha")

pdf = df.toPandas()

In [0]:
sns.countplot(data=pdf, x="tipo_op")
plt.title("Distribución de operaciones por tipo")
plt.show()

In [0]:
df_canal = spark.table("gold_byma.dim_canal").toPandas()

pdf = pdf.merge(df_canal, on="canal_sk", how="left")

sns.countplot(data=pdf, x="nombre_canal")
plt.xticks(rotation=45)
plt.title("Operaciones por canal")
plt.show()

In [0]:
pdf["fecha"] = pd.to_datetime(pdf["fecha_sk"])

df_time = pdf.groupby("fecha").agg({
    "cantidad": "sum",
    "monto_total_usd": "sum"
}).reset_index()

plt.figure(figsize=(10,5))
plt.plot(df_time["fecha"], df_time["monto_total_usd"])
plt.title("Evolución del volumen operado")
plt.show()

In [0]:
top_inst = pdf.groupby("instrumento_sk").agg({
    "cantidad": "sum",
    "monto_total_usd": "sum"
}).sort_values("monto_total_usd", ascending=False).head(10)

top_inst["monto_total_usd"].plot(kind="bar")
plt.title("Top 10 instrumentos")
plt.show()

In [0]:
ops_cliente.sort_values(ascending=False).head(10)

In [0]:

df_fact = spark.table("gold_byma.fact_operaciones")
df_fecha = spark.table("gold_byma.dim_fecha")

df_week = df_fact \
    .join(df_fecha, on="fecha_sk", how="left") \
    .withColumn("semana", date_trunc("week", col("fecha"))) \
    .groupBy("semana") \
    .agg(
        sum("cantidad").alias("cantidad_total"),
        sum("monto_total_usd").alias("monto_total_usd")
    ) \
    .orderBy("semana")

pdf_week = df_week.toPandas()

In [0]:
fig, ax1 = plt.subplots(figsize=(12,5))

ax1.plot(pdf_week["semana"], pdf_week["cantidad_total"], label="Cantidad")
ax1.set_ylabel("Cantidad")
ax1.set_xlabel("Semana")

ax2 = ax1.twinx()
ax2.plot(pdf_week["semana"], pdf_week["monto_total_usd"], linestyle="--")
ax2.set_ylabel("Monto USD")

plt.title("Evolución semanal del volumen operado")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
df_fact_ars = df_fact \
    .join(dim_fecha, "fecha_sk") \
    .join(df_dolar, "fecha", "left") \
    .withColumn(
        "monto_total_ars",
        col("monto_total_usd") * col("ccl")
    )

top_volumen_clean = df_fact_ars.groupBy("instrumento_sk") \
    .agg(
        sum("cantidad").alias("volumen_total")
    ) \
    .orderBy(col("volumen_total").desc()) \
    .limit(10)

top_monto_clean = df_fact_ars.groupBy("instrumento_sk") \
    .agg(
        sum("monto_total_ars").alias("monto_total_ars")
    ) \
    .orderBy(col("monto_total_ars").desc()) \
    .limit(10)

In [0]:
top_volumen_final = top_volumen_clean.join(dim_inst, "instrumento_sk")
top_monto_final = top_monto_clean.join(dim_inst, "instrumento_sk")
pdf_vol = top_volumen_final.toPandas()
pdf_monto = top_monto_final.toPandas()

In [0]:
# volumen
pdf_vol.sort_values("volumen_total").plot(
    kind="barh",
    x="simbolo",
    y="volumen_total",
    title="Top 10 instrumentos por volumen"
)
plt.tight_layout()
plt.show()


In [0]:
pdf_monto.sort_values("monto_total_ars").plot(
    kind="barh",
    x="simbolo",
    y="monto_total_ars",
    title="Top 10 instrumentos por monto total (ARS)"
)
plt.tight_layout()
plt.show()

In [0]:
df_ops_cliente = df_fact.groupBy("cliente_sk") \
    .count() \
    .withColumnRenamed("count", "ops_cliente")

pdf_ops = df_ops_cliente.toPandas()

mediana = pdf_ops["ops_cliente"].median()
media = pdf_ops["ops_cliente"].mean()
p95 = pdf_ops["ops_cliente"].quantile(0.95)
p99 = pdf_ops["ops_cliente"].quantile(0.99)

print(f"Mediana: {mediana}")
print(f"Media: {media:.2f}")
print(f"P95: {p95}")
print(f"P99: {p99}")

In [0]:
# ¿hay clientes hiper-activos? ¿cuál es la mediana?

clientes_hiper = pdf_ops[pdf_ops["ops_cliente"] >= p95]

print("Clientes hiper-activos:", len(clientes_hiper))
print("Porcentaje:", len(clientes_hiper) / len(pdf_ops) * 100)

## 📊 Conclusiones y Hallazgos de Negocio

### 1. Predominio de operaciones de compra
Se observa una clara predominancia de operaciones de **compra** frente a las de **venta** (más del doble en volumen de transacciones).

**Implicancia de negocio:**  
Esto sugiere un comportamiento de los clientes más orientado a la **acumulación de activos** que a la rotación o trading activo. Puede ser una oportunidad para:
- Impulsar productos de inversión de largo plazo
- Diseñar estrategias para incentivar la venta (ej: recomendaciones, alertas de toma de ganancias)
- Evaluar si existe fricción en el proceso de venta

---

### 2. Fuerte concentración en canales digitales tradicionales
El canal **Sitio Web Desktop** concentra la mayor cantidad de operaciones, seguido por **Web Responsive**, mientras que **App Mobile** y **API** tienen una participación muy baja.

**Implicancia de negocio:**  
- Existe una fuerte dependencia del canal web tradicional
- Hay una oportunidad clara de crecimiento en **mobile** y **API trading**
- Se podrían impulsar mejoras en la app mobile o campañas de adopción digital
- Riesgo operativo si el canal principal presenta fallas

---

### 3. Alta concentración de volumen en pocos instrumentos
El volumen operado está altamente concentrado en pocos instrumentos (ej: ITA, XLF), con una caída significativa en el resto.

**Implicancia de negocio:**  
- El negocio depende fuertemente de ciertos activos
- Mayor exposición a riesgos de mercado específicos
- Oportunidad para diversificar la oferta y promover otros instrumentos
- Posible sesgo hacia activos internacionales o específicos (analizar perfil del cliente)

---

### 4. Existencia de clientes hiper-activos (cola larga)
La mediana de operaciones por cliente es **1**, mientras que el percentil 99 llega a **10 operaciones**, identificando un grupo reducido (~6.9%) de clientes hiper-activos.

**Implicancia de negocio:**  
- La mayoría de los clientes tiene baja frecuencia de operación
- Un pequeño grupo genera gran parte de la actividad (posibles traders frecuentes)
- Se pueden diseñar estrategias diferenciadas:
  - **Clientes masivos:** educación financiera, engagement
  - **Clientes hiper-activos:** beneficios, menor comisión, herramientas avanzadas

---

### 5. Alta volatilidad en el volumen semanal operado
El volumen operado presenta fluctuaciones importantes semana a semana, con picos marcados y caídas abruptas.

**Implicancia de negocio:**  
- El negocio está influenciado por condiciones externas (mercado, contexto económico)
- Necesidad de monitoreo constante y capacidad de respuesta rápida
- Oportunidad para generar alertas o insights automáticos en semanas de alta actividad
- Importante para planificación de capacidad operativa y sistemas

---